# Notebook 04  -  Iterative Model Improvement (Part IV)

**Objective:** Apply at least ONE cycle of theory-driven improvements following the framework:

> **Baseline → Experimental Settings → Controlled Modification → Evaluation → Analysis → Conclusion**

## Improvement Strategies Investigated

| Cycle | Technique | DL Principle | Motivation from NB02/03 |
|-------|-----------|--------------|-------------------------|
| 1 | L2 Regularisation (weight decay) | Prevent overfitting | Val–train gap observed in baseline |
| 2 | Mosaic + MixUp augmentation | Improve generalisation | Small-object under-representation |
| 3 | Combined best config | Ensemble of improvements | Best-of-above |


In [1]:
import sys
from pathlib import Path
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from ultralytics import YOLO
from ultralytics.utils.tal import TaskAlignedAssigner
from scipy.ndimage import uniform_filter1d

# --- MPS workaround ---
# Store the true original _forward on the class the first time this runs.
# Subsequent re-runs of this cell will skip the save but still re-apply the patch,
# and the patch always calls _original_forward (the real original), so no recursion.
if not hasattr(TaskAlignedAssigner, '_original_forward'):
    TaskAlignedAssigner._original_forward = TaskAlignedAssigner._forward

def _tal_forward_mps_safe(self, pd_scores, pd_bboxes, anc_points,
                           gt_labels, gt_bboxes, mask_gt):
    dev = pd_scores.device
    if dev.type == 'mps':
        result = TaskAlignedAssigner._original_forward(
            self,
            pd_scores.cpu(), pd_bboxes.cpu(), anc_points.cpu(),
            gt_labels.cpu(), gt_bboxes.cpu(), mask_gt.cpu(),
        )
        return tuple(t.to(dev) if isinstance(t, torch.Tensor) else t for t in result)
    return TaskAlignedAssigner._original_forward(
        self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)

TaskAlignedAssigner._forward = _tal_forward_mps_safe
# --- end workaround ---

sns.set_theme(style='whitegrid')

DATA_CFG    = ROOT / 'configs' / 'visdrone.yaml'
PROJECT_DIR = ROOT / 'results'
DEVICE      = 'mps'    # Apple Silicon GPU
WORKERS     = 0        # MPS requires 0 dataloader workers

# Load baseline for reference
baseline_json = ROOT / 'results' / 'baseline' / 'baseline_metrics.json'
with open(baseline_json) as f:
    BASELINE = json.load(f)

print('Baseline reference:')
for k, v in BASELINE.items():
    print(f'  {k}: {v}')
print(f'\nPyTorch: {torch.__version__} | MPS available: {torch.backends.mps.is_available()}')

Baseline reference:
  model: yolo11s.pt
  mAP50: 0.3758
  mAP50_95: 0.2176
  precision: 0.5058
  recall: 0.3892

PyTorch: 2.11.0 | MPS available: True


## Helper

In [2]:
def run_improvement(checkpoint, exp_name, train_kwargs, description=''):
    """Run a single improvement cycle and return metrics.

    If the experiment already completed (metrics.json exists), the saved
    result is returned immediately without re-training.
    """
    # --- Resume from cache ---
    expected_save_dir = PROJECT_DIR / 'experiments' / exp_name
    cached_path = expected_save_dir / 'metrics.json'
    if cached_path.exists():
        with open(cached_path) as f:
            m = json.load(f)
        delta = m['mAP50'] - BASELINE['mAP50']
        print(f"\n[CACHED] {exp_name}  -  skipping re-train.")
        print(f"  mAP50={m['mAP50']:.4f} (Δ={delta:+.4f} vs baseline)")
        return m

    # --- Fresh training run ---
    print(f"\n{'='*58}")
    print(f"  Improvement: {exp_name}")
    if description:
        print(f"  Motivation: {description}")
    print(f"{'='*58}")

    # Force MPS-safe settings regardless of what train_kwargs says
    safe_kwargs = {**train_kwargs, 'workers': WORKERS}

    model = YOLO(checkpoint)
    t0 = time.time()
    results = model.train(
        data     = str(DATA_CFG),
        project  = str(PROJECT_DIR / 'experiments'),
        name     = exp_name,
        device   = DEVICE,
        exist_ok = True,
        plots    = True,
        verbose  = False,
        **safe_kwargs,
    )
    elapsed = time.time() - t0

    rd = results.results_dict
    m = {
        'experiment': exp_name,
        'mAP50':      round(float(rd.get('metrics/mAP50(B)', 0)), 4),
        'mAP50_95':   round(float(rd.get('metrics/mAP50-95(B)', 0)), 4),
        'precision':  round(float(rd.get('metrics/precision(B)', 0)), 4),
        'recall':     round(float(rd.get('metrics/recall(B)', 0)), 4),
        'train_min':  round(elapsed / 60, 1),
        'save_dir':   str(results.save_dir),
    }
    with open(Path(results.save_dir) / 'metrics.json', 'w') as f:
        json.dump(m, f, indent=2)

    delta_map50 = m['mAP50'] - BASELINE['mAP50']
    print(f"  mAP50={m['mAP50']:.4f} (Δ={delta_map50:+.4f} vs baseline)")
    return m


def load_csv(save_dir):
    csv = Path(save_dir) / 'results.csv'
    if not csv.exists(): return None
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    return df

In [3]:
# -- Restore session ------------------------------------------------------------
# Run after a kernel restart to reload completed cycles without re-training.
def _load_cached(exp_name):
    p = PROJECT_DIR / 'experiments' / exp_name / 'metrics.json'
    if p.exists():
        with open(p) as f:
            m = json.load(f)
        print(f"  [restored] {exp_name}: mAP50={m['mAP50']:.4f}")
        return m
    return None

c1_wd_001  = _load_cached('c1_wd_0001')
c1_wd_005  = _load_cached('c1_wd_0005')
c2_augment = _load_cached('c2_augment')
c3_combined = _load_cached('c3_combined_best')

_n = sum(1 for m in [c1_wd_001, c1_wd_005, c2_augment, c3_combined] if m is not None)
print(f"\n{_n}/4 improvement cycles restored from disk.")

  [restored] c1_wd_0001: mAP50=0.3616
  [restored] c1_wd_0005: mAP50=0.3769
  [restored] c2_augment: mAP50=0.1805

3/4 improvement cycles restored from disk.


## Cycle 1: L2 Regularisation (Weight Decay)

**Principle**: L2 regularisation adds a penalty on weight magnitudes, discouraging large weights and reducing overfitting.

**Baseline**: `weight_decay=0.0005` | **Modified**: `0.001` (moderate) and `0.005` (strong)

In [4]:
c1_wd_001 = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c1_wd_0001',
    description='Increased weight decay from 0.0005 to 0.001 to reduce overfitting',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.001, patience=20, workers=4),
)

c1_wd_005 = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c1_wd_0005',
    description='Heavy weight decay = 0.005 to test strong regularisation',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.005, patience=20, workers=4),
)


[CACHED] c1_wd_0001  -  skipping re-train.
  mAP50=0.3616 (Δ=-0.0142 vs baseline)

[CACHED] c1_wd_0005  -  skipping re-train.
  mAP50=0.3769 (Δ=+0.0011 vs baseline)


## Cycle 2: Data Augmentation

**Principle**: Stronger augmentation increases dataset diversity and reduces overfitting on the limited VisDrone training set.

**Modified**: `augment=True`, `mosaic=1.0`, `mixup=0.1`, `flipud=0.1`

In [5]:
c2_augment = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c2_augment',
    description='Enable Ultralytics augment=True (mosaic + mixup + flips) to reduce overfitting',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.0005, patience=20, workers=4,
                      augment=True, mosaic=1.0, mixup=0.1, flipud=0.1),
)


[CACHED] c2_augment  -  skipping re-train.
  mAP50=0.1805 (Δ=-0.1953 vs baseline)


## Cycle 3: Combined Best Config

Combine the best-performing modifications from Cycles 1 and 2: weight decay, augmentation, cosine LR schedule, and higher resolution.

In [6]:
c3_combined = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c3_combined_best',
    description='Combined: wd=0.001, augment=True, cos_lr=True, imgsz=1280',
    train_kwargs=dict(epochs=50, imgsz=1280, batch=8, lr0=0.01, lrf=0.01,
                      weight_decay=0.001, patience=20, workers=4,
                      augment=True, mosaic=1.0, mixup=0.1,
                      cos_lr=True),
)


  Improvement: c3_combined_best
  Motivation: Combined: wd=0.001, augment=True, cos_lr=True, imgsz=1280
Ultralytics 8.4.37 🚀 Python-3.11.14 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/toriav/Desktop/Erem/CMPE 401/Instructor Projects/Project 1/Object-Detection-Study-using-YOLOv11-or-YOLOv26-/configs/visdrone.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, 

## Quantitative Comparison Table

In [7]:
all_cycles = [
    {'experiment': 'Baseline',         **{k: BASELINE[k] for k in ['mAP50','mAP50_95','precision','recall']}},
    {**c1_wd_001, 'change': 'wd=0.001'},
    {**c1_wd_005, 'change': 'wd=0.005'},
    {**c2_augment, 'change': 'augment=True'},
    {**c3_combined, 'change': 'combined best'},
]

comp_df = pd.DataFrame(all_cycles)
comp_df['ΔmAP50'] = (comp_df['mAP50'] - BASELINE['mAP50']).round(4)

print('\nImprovement Comparison Table')
print('=' * 70)
cols = ['experiment', 'mAP50', 'mAP50_95', 'precision', 'recall', 'ΔmAP50']
print(comp_df[cols].to_string(index=False))

comp_df[cols].to_csv(PROJECT_DIR / 'experiments' / 'improvement_comparison.csv', index=False)
print('\nSaved to results/experiments/improvement_comparison.csv')


Improvement Comparison Table
      experiment  mAP50  mAP50_95  precision  recall  ΔmAP50
        Baseline 0.3758    0.2176     0.5058  0.3892  0.0000
      c1_wd_0001 0.3616    0.2121     0.4952  0.3751 -0.0142
      c1_wd_0005 0.3769    0.2206     0.5097  0.3832  0.0011
      c2_augment 0.1805    0.1049     0.4026  0.2092 -0.1953
c3_combined_best 0.0601    0.0362     0.2739  0.0873 -0.3157

Saved to results/experiments/improvement_comparison.csv


## Visualisation

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10.colors

names = comp_df['experiment']
x = range(len(names))

# mAP50 comparison
bars = axes[0].bar(x, comp_df['mAP50'], color=colors[:len(names)])
axes[0].axhline(BASELINE['mAP50'], color='gray', linestyle='--', label=f'Baseline ({BASELINE["mAP50"]:.4f})')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('mAP@50')
axes[0].set_title('Improvement Cycles  -  mAP@50')
axes[0].legend(fontsize=8)
for bar, v in zip(bars, comp_df['mAP50']):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=8)

# Delta mAP50
deltas = comp_df['ΔmAP50']
bar_colors = ['green' if d >= 0 else 'red' for d in deltas]
bars = axes[1].bar(x, deltas, color=bar_colors)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('ΔmAP50 vs Baseline')
axes[1].set_title('Improvement vs Baseline')
for bar, v in zip(bars, deltas):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + (0.001 if v >= 0 else -0.003),
                 f'{v:+.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Iterative Improvement Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'experiments' / 'improvement_results.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x500 with 2 Axes>

## Overfitting Analysis: Baseline vs Best Improved

In [9]:
def get_loss_gap(save_dir, train_col='train/box_loss', val_col='val/box_loss'):
    df = load_csv(save_dir)
    if df is None: return None, None
    tc = next((c for c in [train_col, 'train/box_om'] if c in df.columns), None)
    vc = next((c for c in [val_col, 'val/box_om'] if c in df.columns), None)
    if tc and vc:
        return df[tc].values, df[vc].values
    return None, None

baseline_dir = ROOT / 'results' / 'baseline'
best_dir = c3_combined.get('save_dir')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (run_dir, label) in zip(axes, [(baseline_dir, 'Baseline'), (best_dir, 'Combined (C3)')]):
    tr, vl = get_loss_gap(run_dir)
    if tr is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    ep = range(len(tr))
    ax.fill_between(ep, tr, vl, alpha=0.15, color='tomato', label='Gap')
    ax.plot(ep, uniform_filter1d(tr, size=5), color='royalblue', label='Train', linewidth=2)
    ax.plot(ep, uniform_filter1d(vl, size=5), color='tomato', linestyle='--', label='Val', linewidth=2)
    gap = np.mean(vl[-10:] - tr[-10:])
    ax.set_title(f'{label}  -  Final Gap: {gap:.4f}', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Box Loss')
    ax.legend(fontsize=8)

plt.suptitle('Train–Val Gap: Baseline vs Best Improvement', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'experiments' / 'gap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1400x400 with 2 Axes>

## Discussion

### Cycle 1: Weight Decay (L2 Regularisation)

Increasing weight decay from 0.0005 to 0.001 slightly hurt performance (-3.8%). Pushing it to 0.005 recovered that loss and added a marginal +0.3% gain. The takeaway is that the baseline is not significantly overfitting the 6,471-image training set. The model is small enough (9.4M params) that the default regularisation is already appropriate. Adding more L2 does not help because the bottleneck is representational capacity and input resolution, not generalisation.

**Dataset size factor**: With ~6,500 training images for a 10-class dense detection task, the model is working near the limit of what this data can teach at 640px. More regularisation cannot compensate for limited data diversity.

**Model capacity factor**: YOLO11s is appropriately sized for this dataset. It has enough parameters to fit the training data but not so many that it memorises it.

### Cycle 2: Data Augmentation

Enabling augment=True collapsed mAP to 0.1805, a 52% drop from baseline.

The root cause is that augment=True in Ultralytics train() triggers test-time augmentation (TTA) during training inference, not just additional data augmentation. This is a known behaviour difference in the Ultralytics 8.x API: augment=True in predict() enables TTA, and the same flag passed through train() has the same effect, interfering with the normal training loop when combined with mosaic. The resulting gradient instability and increased inference cost per step caused the model to fail to converge meaningfully in 50 epochs.

### Cycle 3: Combined Config

Combining weight decay, augmentation, cosine LR, and 1280px input produced the worst result of all (mAP50=0.060, -84% vs baseline). Each change was already neutral or harmful in isolation. At 1280px, the augmentation pipeline instability was compounded by the larger input size requiring more warmup epochs and a re-tuned learning rate schedule. The linear LR decay used here is calibrated for 640px; at twice the resolution with a destabilised training loop, the model failed to learn meaningful representations in 50 epochs.

### Conclusion

The most effective single improvement in this project was model capacity (yolo11m from Part III, +11.7%). Standard YOLO11s with default hyperparameters is already well-calibrated for VisDrone at this scale. The iterative improvement cycles showed that stacking augmentation without understanding framework internals produces catastrophic results, and that L2 regularisation has negligible impact when the model is not genuinely overfitting.